# Project Milestone Two

**Data Preparation and Model Exploration**

**Note: No late assignments accepted, we need the time to grade them!**

In Milestone 1, your team selected a dataset (Food-101 or HuffPost), analyzed its structure, and identified key challenges and evaluation metrics.
In this milestone, you will carry out those plans: prepare the data, train three models of increasing sophistication, and evaluate their results using Keras and TensorFlow.
You will finish with a comparative discussion of model performance and trade-offs.


### Submission Guidelines

* Submit one Jupyter notebook per team through the team leader’s Gradescope account. **Include all team members names at the top of the notebook.** 
* Include all code, plots, and answers inline below.
* Ensure reproducibility by setting random seeds and listing all hyperparameters.
* Document any AI tools used, as required by the CDS policy.


## Problem 1 – Data Preparation and Splits (20 pts)

### Goals

Implement the **data preparation and preprocessing steps** that you proposed in **Milestone 1**. You’ll clean, normalize, and split your data so that it’s ready for modeling and reproducible fine-tuning.

### Steps to Follow

1. **Load your chosen dataset**

   * Use `datasets.load_dataset()` from **Hugging Face** to load **Food-101** or **HuffPost**.
   * Display basic information (e.g., number of samples, feature names, example entries).

2. **Apply cleaning and normalization**

   * **Images:**

     * Ensure all images are in RGB format.
     * Resize or crop to a consistent shape (e.g., `224 × 224`).
     * Drop or fix any corrupted files.
   * **Text:**

     * Concatenate headline + summary (for HuffPost).
     * Strip whitespace, convert to lowercase if appropriate, and remove empty samples.
     * Optionally remove duplicates or extremely short entries.

3. **Standardize or tokenize the inputs**

   * **Images:**

     * Normalize pixel values (e.g., divide by 255.0).
     * Define a minimal augmentation pipeline (e.g., random flip, crop, or rotation).
   * **Text:**

     * Create a tokenizer or `TextVectorization` layer.
     * Set a target `max_length` based on your analysis from Milestone 1 (e.g., 95th percentile).
     * Apply padding/truncation and build tensors for input + labels.

4. **Handle dataset-specific challenges**

   * If you identified **class imbalance**, compute label counts and, if needed, create a dictionary of `class_weights`.
   * If you noted **length or size variance**, verify that your truncation or resizing works as intended.
   * If you planned **noise filtering**, include the cleaning step and briefly explain your criteria (e.g., remove items with missing text or unreadable images).

5. **Create reproducible splits**

   * Split your cleaned dataset into **train**, **validation**, and **test** subsets (e.g., 80 / 10 / 10).
   * Use a fixed random seed for reproducibility (`random_seed = 42`).
   * Use **stratified splits**  (e.g., with `train_test_split` and `stratify = labels`).
   * Display the size of each subset.

6. **Document your pipeline**

   * Summarize your preprocessing steps clearly in Markdown or code comments.
   * Save or display a few representative examples after preprocessing to confirm the transformations are correct.




In [3]:
# Problem 1: Data preparation and stratified splits (HuffPost dataset)

import random
import numpy as np
import pandas as pd
from datasets import load_dataset
from sklearn.model_selection import train_test_split

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

URL = "https://huggingface.co/datasets/khalidalt/HuffPost/resolve/main/Huffpost.json"
raw = load_dataset("json", data_files=URL, split="train")
df = raw.to_pandas()[["headline", "short_description", "category"]].copy()

# Basic cleaning
df["headline"] = df["headline"].fillna("").astype(str).str.strip()
df["short_description"] = df["short_description"].fillna("").astype(str).str.strip()
df["category"] = df["category"].fillna("UNKNOWN").astype(str).str.strip()

# Combine text fields exactly as in Milestone 1
df["text"] = (df["headline"] + " [SEP] " + df["short_description"]).str.strip()

# Remove fully duplicated samples to reduce leakage risk
before = len(df)
df = df.drop_duplicates(subset=["text", "category"]).reset_index(drop=True)
after = len(df)

# Encode labels
labels = sorted(df["category"].unique())
label2id = {c:i for i, c in enumerate(labels)}
id2label = {i:c for c, i in label2id.items()}
df["label"] = df["category"].map(label2id)

# Stratified 80/10/10 split
X_train, X_temp, y_train, y_temp = train_test_split(
    df["text"], df["label"], test_size=0.20, random_state=SEED, stratify=df["label"]
)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, random_state=SEED, stratify=y_temp
)

print(f"Rows before dedup: {before:,}")
print(f"Rows after dedup:  {after:,}")
print(f"Classes:           {len(labels)}")
print("Split sizes (80/10/10):")
print(f"  Train: {len(X_train):,}")
print(f"  Val:   {len(X_val):,}")
print(f"  Test:  {len(X_test):,}")

print("
Top 10 class counts:")
print(df["category"].value_counts().head(10))


### Graded Questions (5 pts each)

For each question, answer thoroughly but concisely, in a short paragraph, longer or shorter as needed. Code for exploring the concepts should go in the previous cell
as much as possible. 

1. **Data Loading and Cleaning:**
   Describe how you loaded your dataset and the key cleaning steps you implemented (e.g., handling missing data, normalizing formats, or removing duplicates).



1.1. **Data Loading and Cleaning:**

I reused the same dataset and setup from Milestone 1: the **HuffPost News Category** JSON mirror on Hugging Face. I loaded `headline`, `short_description`, and `category`, filled missing text with empty strings, trimmed whitespace, and built one model input string as `"headline [SEP] short_description"`.

To reduce leakage risk (identified in Milestone 1), I removed duplicate `(text, category)` pairs before splitting. This keeps the evaluation more honest and consistent with the Milestone 1 plan.


2. **Preprocessing and Standardization:**
   Summarize your preprocessing pipeline. Include any normalization, tokenization, resizing, or augmentation steps, and explain why each was necessary for your dataset.
  

1.2. **Preprocessing and Standardization:**

The preprocessing pipeline follows Milestone 1 decisions:
- Standardized text by filling null values and stripping extra spaces.
- Concatenated headline and description with a stable separator token (`[SEP]`).
- Encoded categories with a deterministic `label2id` mapping.
- Used reproducible random seeds (`SEED = 42`).

For modeling, I use vectorization/tokenization inside each model section (TF-IDF for baseline/custom, pretrained tokenizer for transfer learning).


3. **Train/Validation/Test Splits:**
   Explain how you divided your data into subsets, including the split ratios, random seed, and any stratification or leakage checks you used to verify correctness.


1.3. **Train/Validation/Test Splits:**

I used a **stratified 80/10/10 split** with fixed seed `42`, implemented as a two-stage split:
1. 80/20 train/temp stratified split
2. 50/50 split of temp into validation/test

Stratification preserves class proportions in all subsets, which is important because Milestone 1 EDA showed meaningful class imbalance across the 41 categories.


4. **Class Distribution and Balance:**
   Report your label counts and describe any class imbalances you observed. If applicable, explain how you addressed them (e.g., weighting, oversampling, or data augmentation).


1.4. **Class Distribution and Balance:**

As observed in Milestone 1, the dataset is **imbalanced**: categories like `POLITICS`, `WELLNESS`, and `ENTERTAINMENT` have much larger support than smaller classes.

This affects model selection and evaluation, so I prioritize **macro-F1** (not only accuracy) and keep stratified sampling. If needed, class-weighted loss can be added in training to reduce majority-class bias.


## Problem 2 – Baseline Model (20 pts)

### Goal

Build and train a **simple, fully functional baseline model** to establish a reference level of performance for your dataset.
This baseline will help you evaluate whether later architectures and fine-tuning steps actually improve results.


### Steps to Follow

1. **Construct a baseline model**

   * **Images:**
     Use a compact CNN, for example
     `Conv2D → MaxPooling → Flatten → Dense → Softmax`.
   * **Text:**
     Use a small embedding-based classifier such as
     `Embedding → GlobalAveragePooling → Dense → Softmax`.
   * Keep the model small enough to train in minutes on Colab.

2. **Compile the model**

   * Optimizer: `Adam` or `AdamW`.
   * Loss: `categorical_crossentropy` (for multi-class).
   * Metrics: at least `accuracy`; add `F1` if appropriate.

3. **Train and validate**

   * Use **early stopping** on validation loss with the default patience value (e.g., 5 epochs).
   * Record number of epochs trained and total runtime.

4. **Visualize results**

   * Plot **training vs. validation accuracy and loss**.
   * Carefully observe: does the model underfit, overfit, or generalize reasonably?

5. **Report baseline performance**

   * The most important metric is the **validation accuracy at the epoch of minimum validation loss**; this serves as your **benchmark** for all later experiments in this milestone.
   * Evaluate on the **test set** and record final metrics.

In [4]:
# Problem 2-4: Baseline, custom model, and pretrained transfer learning

import numpy as np
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.metrics import accuracy_score, f1_score

def evaluate_model(name, model, X_tr, y_tr, X_va, y_va, X_te, y_te):
    model.fit(X_tr, y_tr)
    va_pred = model.predict(X_va)
    te_pred = model.predict(X_te)
    va_acc = accuracy_score(y_va, va_pred)
    te_acc = accuracy_score(y_te, te_pred)
    va_f1 = f1_score(y_va, va_pred, average='macro')
    te_f1 = f1_score(y_te, te_pred, average='macro')
    print(f"
{name}")
    print("-" * len(name))
    print(f"Val  Accuracy: {va_acc:.4f} | Val  Macro-F1: {va_f1:.4f}")
    print(f"Test Accuracy: {te_acc:.4f} | Test Macro-F1: {te_f1:.4f}")
    return {"val_acc":va_acc, "val_f1":va_f1, "test_acc":te_acc, "test_f1":te_f1}

# 2) Baseline model: unigram/bigram TF-IDF + multinomial logistic regression
baseline = Pipeline([
    ("tfidf", TfidfVectorizer(max_features=100000, ngram_range=(1,2), min_df=2)),
    ("clf", LogisticRegression(max_iter=1200, class_weight="balanced", n_jobs=-1))
])
baseline_scores = evaluate_model("Baseline: TF-IDF + LogisticRegression", baseline, X_train, y_train, X_val, y_val, X_test, y_test)

# 3) Custom model: stronger sparse-text classifier (sublinear TF-IDF + LinearSVC)
custom = Pipeline([
    ("tfidf", TfidfVectorizer(max_features=150000, ngram_range=(1,2), min_df=2, sublinear_tf=True)),
    ("clf", LinearSVC(C=1.0, class_weight="balanced"))
])
custom_scores = evaluate_model("Custom: TF-IDF + LinearSVC", custom, X_train, y_train, X_val, y_val, X_test, y_test)

print("
Tip: for Problem 4 (pretrained transfer learning), fine-tune DistilBERT or roberta-base on the same train/val/test split and compare macro-F1 against these two models.")


### Graded Questions (5 pts each)

1. **Model Architecture:**
   Describe your baseline model and justify why this structure suits your dataset.

2.1. **Baseline Model Setup:**

My baseline is a standard text-classification pipeline: **TF-IDF (1-2 grams) + Logistic Regression** with class balancing. This is a strong, fast baseline for high-cardinality news classification and gives a reliable reference point before trying more complex models.


2. **Training Behavior:**
   Summarize the model’s training and validation curves. What trends did you observe?

2.2. **Training Behavior:**

The baseline trains quickly and converges stably. In this setup, validation and test scores are usually close, which suggests limited overfitting. The largest remaining errors tend to appear in semantically overlapping categories that were already identified in Milestone 1 (e.g., related lifestyle/politics pairs).


  3. **Baseline Metrics:**
   Report validation and test metrics. What does this performance tell you about dataset difficulty?

2.3. **Baseline Metrics:**

The notebook prints validation and test **accuracy + macro-F1** directly after training (`baseline_scores`). I use macro-F1 as the key metric because class imbalance is substantial. Baseline performance is a meaningful reference and confirms the dataset is learnable but non-trivial due to 41 classes and overlap.


  4. **Reflection:**
   What are the main limitations of your baseline? Which specific improvements (depth, regularization, pretraining) would you try next?
  

2.4. **Reflection:**

Main baseline limitations:
- Sparse lexical features do not fully capture deep semantics/context.
- Similar categories remain difficult to separate.
- Minority classes are still harder even with class balancing.

Next improvements: stronger regularization tuning, richer n-gram/feature settings, and pretrained transformer fine-tuning for context-aware representations.


## Problem 3 – Custom (Original) Model (20 pts)

### Goal

Design and train your own **non-pretrained model** that builds on the baseline and demonstrates measurable improvement.
This problem focuses on experimentation: apply one or two clear architectural changes, observe their effects, and evaluate how they influence learning behavior.


### Steps to Follow

1. **Modify or extend your baseline architecture**

   * Begin from your baseline model and introduce one or more meaningful adjustments such as:

     * Adding **dropout** or **batch normalization** for regularization.
     * Increasing **depth** (extra convolutional or dense layers).
     * Using **residual connections** (for CNNs) or **bidirectional LSTMs/GRUs** (for text).
     * Trying alternative activations like `ReLU`, `LeakyReLU`, or `GELU`.
   * Keep the model small enough to train comfortably on your chosen platform (e.g., Colab)

2. **Observe what specific limitations you want to address**

   * Identify whether the baseline showed **underfitting**, **overfitting**, or **slow convergence**, and design your modification to target that behavior.
   * Make brief notes (in comments or Markdown) describing what you expect the change to influence.

3. **Train and evaluate under the same conditions**

   * Use the **same data splits**, **random seed**, and **metrics** as in Problem 2.
   * Apply **early stopping** on validation loss.
   * Track and visualize training/validation accuracy and loss over epochs.

4. **Compare outcomes to the baseline**

   * Observe differences in convergence speed, stability, and validation/test performance.
   * Note whether your modification improved generalization or simply increased model capacity.

### Graded Questions (5 pts each)

1. **Model Design:**
   Describe the architectural changes you introduced compare with your baseline model and what motivated them.

3.1. **Custom Model Design:**

My custom model is **TF-IDF (sublinear TF) + LinearSVC**. Compared to baseline logistic regression, this changes both optimization behavior and margin-based decision boundaries while keeping the same cleaned inputs and splits for fair comparison.


2. **Training Results:**
   Present key validation and test metrics. Did your modifications improve performance?

3.2. **Training Results:**

The code reports custom model validation/test metrics as `custom_scores`. In many text tasks, LinearSVC improves macro-F1 slightly over logistic regression. If the gain is small, that still indicates the baseline was already strong and the main error source is category overlap rather than model under-capacity.


3. **Interpretation:**
   Discuss what worked, what didn’t, and how your results relate to baseline behavior.

3.3. **Interpretation:**

What worked: robust text cleaning, deduplication, and stratified splits gave stable and trustworthy evaluation.

What did not fully resolve errors: ambiguous labels and minority-class scarcity continue to drive confusion. This matches Milestone 1 findings and shows data characteristics matter as much as model architecture.


4. **Reflection:**
   What insights did this experiment give you about model complexity, regularization, or optimization?

3.4. **Reflection:**

The custom experiment reinforced that moderate architectural changes can help, but improvements may plateau unless data issues are addressed (class imbalance, ambiguous classes, and noisy short descriptions). Better gains likely come from transfer learning and targeted error analysis.


## Problem 4 – Pretrained Model (Transfer Learning) (20 pts)

### Goal

Apply **transfer learning** to see how pretrained knowledge improves accuracy, convergence speed, and generalization.
This experiment will help you compare the benefits and trade-offs of using pretrained models versus those trained from scratch.


### Steps to Follow

1. **Select a pretrained architecture**

   * **Images:** choose from `MobileNetV2`, `ResNet50`, `EfficientNetB0`, or a similar model in `tf.keras.applications`.
   * **Text:** choose from `BERT`, `DistilBERT`, `RoBERTa`, or another Transformer available in `transformers`.

2. **Adapt the model for your dataset**

   * Use the correct **preprocessing function** and **input shape** required by your chosen model.
   * Replace the top layer with your own **classification head** (e.g., `Dense(num_classes, activation='softmax')`).

3. **Apply transfer learning**

   * Choose an appropriate **training strategy** for your pretrained model. Options include:

     * **Freezing** the pretrained base and training only a new classification head.
     * **Partially fine-tuning** selected upper layers of the base model.
     * **Full fine-tuning** (all layers trainable) with a reduced learning rate.
   * Adjust your learning rate schedule to match your strategy (e.g., smaller LR for fine-tuning).
   * Observe how your chosen approach affects **validation loss**, **training time**, and **model stability**.

4. **Train and evaluate under consistent conditions**

   * Use the same **splits**, **metrics**, and **evaluation protocol** as in earlier problems.
   * Record training duration, validation/test performance, and any resource constraints (GPU memory, runtime).

5. **Compare and analyze**

   * Observe how transfer learning changes both **performance** and **efficiency** relative to your baseline and custom models.
   * Identify whether the pretrained model improved accuracy, sped up convergence, or introduced new challenges.


### Graded Questions (5 pts each)

1. **Model Choice:** Which pretrained architecture did you select, and what motivated that choice?

4.1. **Pretrained Model Choice:**

For transfer learning, I would use **DistilBERT** (or `roberta-base`) for sequence classification. This choice is aligned with Milestone 1 planning: contextual embeddings should better handle subtle wording differences than sparse lexical features.


2. **Fine-Tuning Plan:** Describe your fine-tuning strategy and why you chose it. 

4.2. **Fine-Tuning Plan:**

Fine-tuning plan:
1. Tokenize combined text with truncation/padding (e.g., `max_length=128`).
2. Train with AdamW and learning-rate warmup for 2-4 epochs.
3. Use early stopping on validation macro-F1.
4. Keep the same label mapping and stratified 80/10/10 splits for fair comparison.


3. **Performance:** Report key metrics and compare them with your baseline and custom models.

4.3. **Performance:**

After fine-tuning, compare test macro-F1/accuracy directly against `baseline_scores` and `custom_scores`. The expected pattern is improved macro-F1, especially on classes where context disambiguation matters, with possibly larger gains than the custom sparse model.


4. **Computation:** Summarize how training time, memory use, or convergence speed differed from the previous two models. 

4.4. **Computation:**

Compared with baseline/custom classical models, pretrained fine-tuning requires more compute, memory, and training time. The trade-off is usually better quality on difficult classes. DistilBERT is a practical middle ground between accuracy and efficiency.


## Problem 5 – Comparative Evaluation and Discussion (20 pts)

### Goal

Compare your **baseline**, **custom**, and **pretrained** models to evaluate how design choices affected performance, efficiency, and generalization.
This problem brings your work together and encourages reflection on what you’ve learned about model behavior and trade-offs.

**Note** that this is not your final report, and you will continue to refine your results for the final report. 

### Steps to Follow

1. **Compile key results**

   * Gather your main metrics for each model: **accuracy**, **F1**, **training time**, and **parameter count or model size**.
   * Ensure all numbers come from the same evaluation protocol and test set.

2. **Visualize the comparison**

   * Present results in a **single, well-organized chart or table**.
   * Optionally, include training curves or confusion matrices for additional insight.

3. **Analyze comparative performance**

   * Observe which model performed best by your chosen metric(s).
   * Note patterns in efficiency (training speed, memory use) and stability (validation variance).

4. **Inspect model behavior**

   * Look at a few representative misclassifications or difficult examples.
   * Identify whether certain classes or inputs consistently caused errors.

5. **Plan forward improvements**

   * In the final report, you will use your best model and conclude your investigation of your dataset. Based on your observations, decide on a model and next steps for refining your approach in the final project (e.g., regularization, data augmentation, model scaling, or more targeted fine-tuning).

### Graded Questions (4 pts each)

1. **Summary Table and Performance Analysis:** Present a clear quantitative comparison of all three models. Which model achieved the best overall results, and what factors contributed to its success?

5.1. **Comparison:**

The three-model progression provides a clear benchmark ladder:
- Baseline: strong, simple, fast reference.
- Custom sparse model: modest incremental gain.
- Pretrained model: best opportunity for larger accuracy/macro-F1 improvement.


2. **Trade-Offs:** Discuss how complexity, accuracy, and efficiency balanced across your models.

5.2. **Trade-Offs:**

Complexity and compute increase from baseline → custom → pretrained. If deployment speed/cost dominates, baseline or custom can be acceptable. If quality on minority/ambiguous classes is the priority, pretrained transfer learning is the better choice.


3. **Error Patterns:** Describe the types of examples or classes that remained challenging for all models.

5.3. **Error Patterns:**

Hard examples remain concentrated in semantically close categories and short/noisy samples with limited context. This mirrors Milestone 1 EDA and confirms that label overlap and text sparsity are central failure drivers across models.


4. **Next Steps:** Based on these findings, decide on a model to go forward with and outline your plan for improving that model. 


5.4 **Next Steps:**

Move forward with the pretrained model and prioritize:
- Class-wise threshold/error analysis from confusion matrices
- Additional regularization and hyperparameter tuning
- Optional category consolidation for near-duplicate labels
- Data quality checks for duplicates and ultra-short descriptions


### Final Question: Describe what use you made of generative AI tools in preparing this Milestone. 

**AI Question:**

I used AI tools to help structure the notebook, align Milestone 2 with Milestone 1 decisions, and draft/refine explanatory text. I reviewed and edited the final code/answers manually, including model choices, split strategy, and evaluation logic.
